In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")

#client.sample_mflix #온점 표기법 -> 온점 포함 // 숫자로 시작 x 
db = client["sample_mflix"]

users = db.users
comments = db.comments 
movies = db.movies

print(movies)


Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'sample_mflix'), 'movies')


In [11]:
#star 가 포함된 영화 제목 가져오기 

# for movie in movies.find({"title":{"$regex":"star","$options":"i"}},{"_id":0,"title":1}):
#     print(movie)


star_titles = list(movies.find({"title":{"$regex":".*star.*"}},{"_id":0,"title":1}))
star_titles

[{'title': 'Marjorie Morningstar'},
 {'title': 'Jesus Christ Superstar'},
 {'title': 'Firestarter'},
 {'title': 'Konec starych casu'},
 {'title': 'Bastard Out of Carolina'},
 {'title': 'Fègi Is a Bastard'},
 {'title': 'Superstar'},
 {'title': 'Firestarter 2: Rekindled'},
 {'title': 'Firestarter 2: Rekindled'},
 {'title': 'Battlestar Galactica'},
 {'title': 'Restart'},
 {'title': 'Bastards of the Party'},
 {'title': 'Bastards'},
 {'title': 'Beauty and the Bastard'},
 {'title': 'Los bastardos'},
 {'title': 'Battlestar Galactica: Razor'},
 {'title': 'Telstar: The Joe Meek Story'},
 {'title': 'Battlestar Galactica: The Plan'},
 {'title': 'Battlestar Galactica: Blood & Chrome'},
 {'title': 'Mad Bastards'},
 {'title': 'Superstar'},
 {'title': 'Radiostars'},
 {'title': 'Bastards'}]

In [19]:
# 각 감독들이 제작한 영화의 수가 25개 이상인 감독들만 찾아서 출력 '
pipeline = [
    {"$unwind" : "$directors"},
    {"$group" : {
        "_id" : "$directors",
        "movieCount" : {"$sum":1}
    }},
    {"$match":{"movieCount":{"$gte":25}}},
    {"$sort":{"movieCount":-1}}
]

for movie in movies.aggregate(pipeline):
    print(movie["_id"], movie["movieCount"])

# director_count = list(movies.aggregate(pipeline))
# director_count

Woody Allen 40
John Ford 35
Takashi Miike 34
John Huston 34
Werner Herzog 33
Martin Scorsese 32
Alfred Hitchcock 31
Sidney Lumet 30
George Cukor 29
Mario Monicelli 29
Steven Spielberg 29
Michael Apted 29
Robert Altman 28
Steven Soderbergh 28
Spike Lee 28
Ken Loach 27
Wim Wenders 27
Jean-Luc Godard 27
Johnnie To 27
Clint Eastwood 27
William Wyler 26
Michael Winterbottom 26
Ridley Scott 25
Ingmar Bergman 25


In [38]:
# 평점을 기준으로 상위 5개의 영화를 찾기 투표 1000표 이상
pipeline = [
    {"$match":{"imdb.votes":{"$gte":1000}}},
    {"$sort":{"imdb.rating":-1}},
    {"$limit":5}
]

for movie in movies.aggregate(pipeline):
    print(f"제목:{movie["title"]}, 투표 수:{movie["imdb"]["votes"]}, 평점: {movie["imdb"]["rating"]}")

제목:Band of Brothers, 투표 수:183802, 평점: 9.6
제목:Planet Earth, 투표 수:82896, 평점: 9.5
제목:The Civil War, 투표 수:4624, 평점: 9.4
제목:The Civil War, 투표 수:4625, 평점: 9.4
제목:The Shawshank Redemption, 투표 수:1513145, 평점: 9.3


In [42]:
top_rated_movies = list(movies.find({"imdb.votes":{"$gte":1000}}).sort("imdb.rating",-1).limit(5))

for movie in top_rated_movies :
    print(movie["title"], movie["imdb"]["votes"], movie["imdb"]["rating"])

Band of Brothers 183802 9.6
Planet Earth 82896 9.5
The Civil War 4624 9.4
The Civil War 4625 9.4
The Shawshank Redemption 1513145 9.3


In [ ]:
from bs4 import BeautifulSoup
import requests

